# Pipeline

In [55]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


In [48]:
df = pd.read_csv('CAR DETAILS FROM CAR DEKHO.csv')
df.head(2)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner


In [49]:
pipe1 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', StandardScaler(), ['year', 'km_driven']),
        ('cat', OneHotEncoder(sparse_output=False), ['seller_type', 'transmission', 'owner']) # nominal encoding
        ])
    ),
    ('model', LinearRegression())
])
pipe1[:-1].set_output(transform='pandas').fit_transform(df).head(2)

,num__year,num__km_driven,cat__seller_type_Dealer,cat__seller_type_Individual,cat__seller_type_Trustmark Dealer,cat__transmission_Automatic,cat__transmission_Manual,cat__owner_First Owner,cat__owner_Fourth & Above Owner,cat__owner_Second Owner,cat__owner_Test Drive Car,cat__owner_Third Owner
0,-1.445074,0.081139,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
1,-1.445074,-0.347689,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [50]:
ord_enc = OrdinalEncoder(categories=[['Test Drive Car', 'First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner']])
df['owner_enc'] = ord_enc.fit_transform(df[['owner']])
df

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,owner_enc
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,1.0
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,1.0
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,1.0
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,1.0
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,2.0
...,...,...,...,...,...,...,...,...,...
4335,Hyundai i20 Magna 1.4 CRDi (Diesel),2014,409999,80000,Diesel,Individual,Manual,Second Owner,2.0
4336,Hyundai i20 Magna 1.4 CRDi,2014,409999,80000,Diesel,Individual,Manual,Second Owner,2.0
4337,Maruti 800 AC BSIII,2009,110000,83000,Petrol,Individual,Manual,Second Owner,2.0
4338,Hyundai Creta 1.6 CRDi SX Option,2016,865000,90000,Diesel,Individual,Manual,First Owner,1.0


In [51]:
pipe2 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', StandardScaler(), ['year', 'km_driven']),
        ('cat_nom', OneHotEncoder(sparse_output=False), ['seller_type', 'transmission']), # nominal encoding
        ('cat_ord', OrdinalEncoder(categories=[['Test Drive Car', 'First Owner', 'Second Owner', 
                                                'Third Owner', 'Fourth & Above Owner']]), ['owner']) # ordinal encoding

        ])
    ),
    ('model', LinearRegression())
])
pipe2[:-1].set_output(transform='pandas').fit_transform(df).head(2)

,num__year,num__km_driven,cat_nom__seller_type_Dealer,cat_nom__seller_type_Individual,cat_nom__seller_type_Trustmark Dealer,cat_nom__transmission_Automatic,cat_nom__transmission_Manual,cat_ord__owner
0,-1.445074,0.081139,0.0,1.0,0.0,0.0,1.0,1.0
1,-1.445074,-0.347689,0.0,1.0,0.0,0.0,1.0,1.0


In [54]:
pipe3 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', 'passthrough', ['year', 'km_driven']),
        ('cat_nom', OneHotEncoder(sparse_output=False), ['seller_type', 'transmission']), # nominal encoding
        ('cat_ord', OrdinalEncoder(categories=[['Test Drive Car', 'First Owner', 'Second Owner', 
                                                'Third Owner', 'Fourth & Above Owner']]), ['owner']) # ordinal encoding

        ])
    ),
    ('model', LinearRegression())
])
pipe3[:-1].set_output(transform='pandas').fit_transform(df).head(2)

,num__year,num__km_driven,cat_nom__seller_type_Dealer,cat_nom__seller_type_Individual,cat_nom__seller_type_Trustmark Dealer,cat_nom__transmission_Automatic,cat_nom__transmission_Manual,cat_ord__owner
0,2007,70000,0.0,1.0,0.0,0.0,1.0,1.0
1,2007,50000,0.0,1.0,0.0,0.0,1.0,1.0


In [56]:
X = df.drop(columns=['selling_price'])
y = np.log(df['selling_price'])

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipes = {
    'pipe1': pipe1,
    'pipe2': pipe2,
    'pipe3': pipe3
}
for name, pipe in pipes.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f'{name}: R2 Score = {r2_score(np.exp(y_test), np.exp(y_pred))}')
    print(f'{name}: MAE = {mean_absolute_error(np.exp(y_test), np.exp(y_pred))}')

pipe1: R2 Score = 0.35845788306782944
pipe1: MAE = 205760.11824475077
pipe2: R2 Score = 0.35833227979299587
pipe2: MAE = 205815.6990390986
pipe3: R2 Score = 0.35833227979297766
pipe3: MAE = 205815.6990391001
